# 01 - Prepare PrimeKG for Neo4j Ingestion

This notebook converts the raw PrimeKG CSV files into stable, import-ready staging files for Neo4j.

The preparation step is intentionally separate from database loading. It gives the project a durable ingestion boundary: raw PrimeKG files can change, but the backend and graph database loader can depend on a normalized staging shape.

## Database Choice and Ingestion Strategy

Neo4j is the primary graph database target for this project because it is mature, production-oriented, Python/FastAPI friendly, and supports the interactive graph use cases needed by BioKG Explorer:

- shortest paths between biomedical entities
- node-neighborhood expansion
- detailed node and relationship retrieval
- indexed name search
- interactive traversal from a React/D3/Cytoscape frontend
- optional graph algorithms through the Neo4j Graph Data Science plugin

This notebook prepares enriched nodes and deduplicated undirected relationships. The next notebook loads those staging files into Neo4j.

## 1. Imports and Paths

Import the libraries used for local transformation. DuckDB is used for scalable CSV processing so we can deduplicate millions of relationships without loading the whole edge file into pandas.

In [1]:
from pathlib import Path
import json
import re

import duckdb
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "prime-kg-dataset"
BUILD_DIR = Path("build")
NEO4J_IMPORT_DIR = BUILD_DIR / "neo4j_import"

NEO4J_IMPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PrimeKG data: {DATA_DIR}")
print(f"Neo4j staging output: {NEO4J_IMPORT_DIR.resolve()}")

Project root: /Users/faizanfaisal/Laptop_Data/Davis_Courses/ECS273_Visual_Analytics/proj/BioKG-Explorer
PrimeKG data: /Users/faizanfaisal/Laptop_Data/Davis_Courses/ECS273_Visual_Analytics/proj/BioKG-Explorer/prime-kg-dataset
Neo4j staging output: /Users/faizanfaisal/Laptop_Data/Davis_Courses/ECS273_Visual_Analytics/proj/BioKG-Explorer/graph-ingestion/build/neo4j_import


## 2. Define Label and Relationship Normalization

Neo4j labels and relationship types should be stable, readable, and safe for Cypher. The raw PrimeKG node types are preserved as properties, while normalized labels and relationship types are added for database structure.

In [2]:
NODE_LABEL_MAP = {
    "gene/protein": "GeneProtein",
    "disease": "Disease",
    "drug": "Drug",
    "effect/phenotype": "Phenotype",
    "anatomy": "Anatomy",
    "biological_process": "BiologicalProcess",
    "molecular_function": "MolecularFunction",
    "cellular_component": "CellularComponent",
    "pathway": "Pathway",
    "exposure": "Exposure",
}

RELATION_TYPE_OVERRIDES = {
    "off-label use": "OFF_LABEL_USE",
}

def to_cypher_rel_type(relation: str) -> str:
    if relation in RELATION_TYPE_OVERRIDES:
        return RELATION_TYPE_OVERRIDES[relation]
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(relation)).strip("_").upper()
    if not value:
        raise ValueError(f"Cannot convert relation to relationship type: {relation!r}")
    if value[0].isdigit():
        value = f"REL_{value}"
    return value

def clean_text(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip()
        return value if value else None
    return value

NODE_LABEL_MAP

{'gene/protein': 'GeneProtein',
 'disease': 'Disease',
 'drug': 'Drug',
 'effect/phenotype': 'Phenotype',
 'anatomy': 'Anatomy',
 'biological_process': 'BiologicalProcess',
 'molecular_function': 'MolecularFunction',
 'cellular_component': 'CellularComponent',
 'pathway': 'Pathway',
 'exposure': 'Exposure'}

## 3. Load and Inspect Nodes

Load the canonical PrimeKG node table. Every graph node will receive the stable unique property `primekg_index`, which maps to `node_index` in the source dataset.

In [3]:
nodes = pd.read_csv(DATA_DIR / "nodes.csv")

nodes = nodes.rename(
    columns={
        "node_index": "primekg_index",
        "node_id": "source_node_id",
        "node_type": "node_type",
        "node_name": "name",
        "node_source": "node_source",
    }
)

nodes["app_label"] = nodes["node_type"].map(NODE_LABEL_MAP)
missing_labels = nodes[nodes["app_label"].isna()]["node_type"].unique()
if len(missing_labels):
    raise ValueError(f"Missing Neo4j label mappings for node types: {missing_labels}")

nodes["name_lc"] = nodes["name"].str.lower()
nodes["source_node_id"] = nodes["source_node_id"].astype(str)

print(nodes.shape)
nodes.head()

(129375, 7)


,primekg_index,source_node_id,node_type,name,node_source,app_label,name_lc
0,0,9796,gene/protein,PHYHIP,NCBI,GeneProtein,phyhip
1,1,7918,gene/protein,GPANK1,NCBI,GeneProtein,gpank1
2,2,8233,gene/protein,ZRSR2,NCBI,GeneProtein,zrsr2
3,3,4899,gene/protein,NRF1,NCBI,GeneProtein,nrf1
4,4,5297,gene/protein,PI4KA,NCBI,GeneProtein,pi4ka


## 4. Aggregate Disease Features

Disease features may contain repeated `node_index` rows. This cell collapses them to one row per node by taking the first non-empty value per column, then prefixes feature columns so they are clear in Neo4j.

In [4]:
def first_non_null(series):
    non_null = series.dropna()
    if non_null.empty:
        return None
    return non_null.iloc[0]

disease_features = pd.read_csv(DATA_DIR / "disease_features.csv")
disease_features = disease_features.rename(columns={"node_index": "primekg_index"})

feature_columns = [column for column in disease_features.columns if column != "primekg_index"]
disease_features_agg = (
    disease_features.groupby("primekg_index", as_index=False)[feature_columns]
    .agg(first_non_null)
)

disease_features_agg = disease_features_agg.rename(
    columns={column: f"disease_{column}" for column in disease_features_agg.columns if column != "primekg_index"}
)

print(disease_features.shape, "raw disease feature rows")
print(disease_features_agg.shape, "aggregated disease feature rows")
disease_features_agg.head()

(44133, 18) raw disease feature rows
(17080, 18) aggregated disease feature rows


,primekg_index,disease_mondo_id,disease_mondo_name,disease_group_id_bert,disease_group_name_bert,disease_mondo_definition,disease_umls_description,disease_orphanet_definition,disease_orphanet_prevalence,disease_orphanet_epidemiology,disease_orphanet_clinical_description,disease_orphanet_management_and_treatment,disease_mayo_symptoms,disease_mayo_causes,disease_mayo_risk_factors,disease_mayo_complications,disease_mayo_prevention,disease_mayo_see_doc
0,27158,13924,osteogenesis imperfecta type 13,13924_12592_14672_13460_12591_12536_30861_8146...,osteogenesis imperfecta,Any osteogenesis imperfecta in which the cause...,"collagen diseases characterized by brittle, os...",A moderate form of osteogenesis imperfecta cha...,<1/1000000,The overall prevalence of OI is estimated at b...,"There are three subtypes of OI type II (A, B a...",Management should be multidisciplinary involvi...,NaN,NaN,NaN,NaN,NaN,NaN
1,27159,11160,autosomal recessive nonsyndromic deafness 15,11160_13119_13978_12060_12327_12670_13210_1106...,autosomal recessive nonsyndromic deafness,Any autosomal recessive nonsyndromic deafness ...,A bilateral form of sensorineural hearing impa...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27160,8099,congenital stationary night blindness autosoma...,8099_12497_12498,congenital stationary night blindness autosoma...,Any congenital stationary night blindness in w...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27161,14854,autosomal dominant nonsyndromic deafness 66,14854_14293_14470_12380_11832_14603_14853_1176...,autosomal dominant nonsyndromic deafness,Any autosomal dominant nonsyndromic deafness i...,A bilateral form of sensorineural hearing impa...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,27162,33202,"deafness, autosomal recessive 109",33202_32776_30905_33670_33200_32740_32732_3320...,"deafness, autosomal recessive",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Load Drug Features

Drug features are already keyed by `node_index`. The columns are prefixed before merging so drug properties are easy to distinguish from generic node properties.

In [5]:
drug_features = pd.read_csv(DATA_DIR / "drug_features.csv")
drug_features = drug_features.rename(columns={"node_index": "primekg_index"})
drug_features = drug_features.rename(
    columns={column: f"drug_{column}" for column in drug_features.columns if column != "primekg_index"}
)

print(drug_features.shape)
drug_features.head()

(7957, 18)


,primekg_index,drug_description,drug_half_life,drug_indication,drug_mechanism_of_action,drug_protein_binding,drug_pharmacodynamics,drug_state,drug_atc_1,drug_atc_2,drug_atc_3,drug_atc_4,drug_category,drug_group,drug_pathway,drug_molecular_weight,drug_tpsa,drug_clogp
0,14012,Copper is a transition metal and a trace eleme...,NaN,For use in the supplementation of total parent...,Copper is absorbed from the gut via high affin...,Copper is nearly entirely bound by ceruloplasm...,Copper is incorporated into many enzymes throu...,Copper is a solid.,NaN,NaN,NaN,NaN,Copper is part of Copper-containing Intrauteri...,Copper is approved and investigational.,NaN,NaN,NaN,NaN
1,14013,Oxygen is an element displayed by the symbol O...,The half-life is approximately 122.24 seconds,Oxygen therapy in clinical settings is used ac...,Oxygen therapy increases the arterial pressure...,Oxygen binds to oxygen-carrying protein in red...,Oxygen therapy improves effective cellular oxy...,Oxygen is a gas.,Oxygen is anatomically related to various.,Oxygen is in the therapeutic group of all othe...,Oxygen is pharmacologically related to all oth...,The chemical and functional group of is medic...,Oxygen is part of Chalcogens ; Elements ; Gase...,Oxygen is approved and vet_approved.,NaN,The molecular weight is 32.0.,Oxygen has a topological polar surface area of...,NaN
2,14014,"Flunisolide (marketed as AeroBid, Nasalide, Na...",The half-life is 1.8 hours,For the maintenance treatment of asthma as a p...,Flunisolide is a glucocorticoid receptor agoni...,Approximately 40% after oral inhalation,Flunisolide is a synthetic corticosteroid. It ...,Flunisolide is a solid.,Flunisolide is anatomically related to respira...,Flunisolide is in the therapeutic group of nas...,Flunisolide is pharmacologically related to de...,The chemical and functional group of is corti...,Flunisolide is part of Adrenal Cortex Hormones...,Flunisolide is approved and investigational.,NaN,The molecular weight is 434.5.,Flunisolide has a topological polar surface ar...,The log p value of is 2.41.
3,14015,Alclometasone is synthetic glucocorticoid ster...,NaN,For the relief of the inflammatory and pruriti...,The mechanism of the anti-inflammatory activit...,NaN,Alclometasone is a synthetic corticosteroid fo...,Alclometasone is a solid.,Alclometasone is anatomically related to derma...,Alclometasone is in the therapeutic group of c...,Alclometasone is pharmacologically related to ...,The chemical and functional group of is corti...,Alclometasone is part of Adrenal Cortex Hormon...,Alclometasone is approved.,NaN,NaN,NaN,NaN
4,14016,Medrysone is a corticosteroid used in ophthalm...,NaN,"For the treatment of allergic conjunctivitis, ...",There is no generally accepted explanation for...,NaN,Medrysone is a topical anti-inflammatory corti...,Medrysone is a solid.,Medrysone is anatomically related to sensory o...,Medrysone is in the therapeutic group of ophth...,Medrysone is pharmacologically related to anti...,The chemical and functional group of is corti...,Medrysone is part of Adrenal Cortex Hormones ;...,Medrysone is approved.,NaN,The molecular weight is 344.5.,Medrysone has a topological polar surface area...,The log p value of is 3.36.


## 6. Add Disease Grouping Metadata

PrimeKG includes disease grouping files. These group names are useful for aggregate visualizations and for reducing visual clutter during disease-centered exploration.

In [6]:
grouped_diseases = pd.read_csv(DATA_DIR / "kg_grouped_diseases.csv")
grouped_diseases = grouped_diseases.rename(
    columns={
        "node_id": "source_node_id",
        "group_name_auto": "disease_group_name_auto",
        "group_name_bert": "disease_group_name_bert",
    }
)
grouped_diseases["source_node_id"] = grouped_diseases["source_node_id"].astype(str)
grouped_diseases = grouped_diseases[["source_node_id", "disease_group_name_auto", "disease_group_name_bert"]].drop_duplicates("source_node_id")

bert_map = pd.read_csv(DATA_DIR / "kg_grouped_diseases_bert_map.csv")
bert_map = bert_map.rename(
    columns={
        "node_id": "source_node_id",
        "group_id_bert": "disease_group_id_bert",
    }
)
bert_map["source_node_id"] = bert_map["source_node_id"].astype(str)
bert_map = bert_map[["source_node_id", "disease_group_id_bert"]].drop_duplicates("source_node_id")

print(grouped_diseases.shape)
print(bert_map.shape)
grouped_diseases.head()

(22205, 3)
(6392, 2)


,source_node_id,disease_group_name_auto,disease_group_name_bert
0,13924,osteogenesis imperfecta,osteogenesis imperfecta
1,11160,autosomal recessive nonsyndromic deafness,autosomal recessive nonsyndromic deafness
2,8099,congenital stationary night blindness autosoma...,congenital stationary night blindness autosoma...
3,14854,autosomal dominant nonsyndromic deafness,autosomal dominant nonsyndromic deafness
4,33202,"deafness, autosomal recessive","deafness, autosomal recessive"


## 7. Build Enriched Node Staging Table

Merge all available local enrichment into one node staging file. This file preserves PrimeKG identifiers and source provenance while adding clinical/drug/group metadata for detail panels and search.

In [7]:
nodes_enriched = (
    nodes.merge(disease_features_agg, on="primekg_index", how="left")
    .merge(drug_features, on="primekg_index", how="left")
    .merge(grouped_diseases, on="source_node_id", how="left", suffixes=("", "_grouped"))
    .merge(bert_map, on="source_node_id", how="left", suffixes=("", "_bert_map"))
)

# Coalesce overlapping group fields from disease_features and grouping files into stable columns.
for canonical_column in ["disease_group_name_bert", "disease_group_id_bert"]:
    matching_columns = [
        column
        for column in nodes_enriched.columns
        if column == canonical_column or column.startswith(f"{canonical_column}_")
    ]
    if not matching_columns:
        continue

    nodes_enriched[canonical_column] = nodes_enriched[matching_columns].bfill(axis=1).iloc[:, 0]
    duplicate_columns = [column for column in matching_columns if column != canonical_column]
    nodes_enriched = nodes_enriched.drop(columns=duplicate_columns)

search_columns = [
    "name",
    "node_type",
    "node_source",
    "disease_group_name_auto",
    "disease_group_name_bert",
    "disease_mondo_name",
    "drug_description",
    "drug_indication",
]

nodes_enriched["search_text"] = (
    nodes_enriched.reindex(columns=search_columns, fill_value="")
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

for column in nodes_enriched.columns:
    if nodes_enriched[column].dtype == "object":
        nodes_enriched[column] = nodes_enriched[column].map(clean_text)

nodes_enriched = nodes_enriched.where(pd.notna(nodes_enriched), None)

print(nodes_enriched.shape)
nodes_enriched.head()

(129375, 43)


,primekg_index,source_node_id,node_type,name,node_source,app_label,name_lc,disease_mondo_id,disease_mondo_name,disease_group_id_bert,...,drug_atc_3,drug_atc_4,drug_category,drug_group,drug_pathway,drug_molecular_weight,drug_tpsa,drug_clogp,disease_group_name_auto,search_text
0,0,9796,gene/protein,PHYHIP,NCBI,GeneProtein,phyhip,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ornithine aminotransferase deficiency,phyhip gene/protein ncbi ornithine aminotransf...
1,1,7918,gene/protein,GPANK1,NCBI,GeneProtein,gpank1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,microcephaly with or without chorioretinopathy...,gpank1 gene/protein ncbi microcephaly with or ...
2,2,8233,gene/protein,ZRSR2,NCBI,GeneProtein,zrsr2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,phaeochromocytoma,zrsr2 gene/protein ncbi phaeochromocytoma phae...
3,3,4899,gene/protein,NRF1,NCBI,GeneProtein,nrf1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,monofixation syndrome,nrf1 gene/protein ncbi monofixation syndrome m...
4,4,5297,gene/protein,PI4KA,NCBI,GeneProtein,pi4ka,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,urethritis (disease),pi4ka gene/protein ncbi urethritis (disease) u...


## 8. Write Enriched Nodes CSV

Write the node staging file. The Neo4j loader notebook reads this file in chunks and assigns both a generic `Entity` label and a type-specific label such as `Disease`, `Drug`, or `GeneProtein`.

In [8]:
nodes_output = NEO4J_IMPORT_DIR / "nodes_enriched.csv"
nodes_enriched.to_csv(nodes_output, index=False)

print(f"Wrote {len(nodes_enriched):,} enriched nodes to {nodes_output}")

Wrote 129,375 enriched nodes to build/neo4j_import/nodes_enriched.csv


## 9. Deduplicate Undirected Relationships

PrimeKG stores undirected relationships with reciprocal rows. This step stores each undirected edge once using `source_index = min(x_index, y_index)` and `target_index = max(x_index, y_index)`. The original relation and display relation are preserved as properties.

In [9]:
edges_path = DATA_DIR / "edges.csv"
relationships_output = NEO4J_IMPORT_DIR / "relationships_undirected.csv"

con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=4")

query = f"""
COPY (
    SELECT
        relation,
        display_relation,
        LEAST(CAST(x_index AS BIGINT), CAST(y_index AS BIGINT)) AS source_index,
        GREATEST(CAST(x_index AS BIGINT), CAST(y_index AS BIGINT)) AS target_index,
        COUNT(*) AS primekg_row_count
    FROM read_csv_auto('{edges_path.as_posix()}', HEADER=TRUE)
    WHERE CAST(x_index AS BIGINT) <> CAST(y_index AS BIGINT)
    GROUP BY relation, display_relation, source_index, target_index
    ORDER BY relation, source_index, target_index
) TO '{relationships_output.as_posix()}' (HEADER, DELIMITER ',');
"""

con.execute(query)
print(f"Wrote undirected relationship staging file to {relationships_output}")

Wrote undirected relationship staging file to build/neo4j_import/relationships_undirected.csv


## 10. Create Relationship Type Map

Create a small mapping file from PrimeKG relation names to Neo4j relationship types. The loader uses this mapping to create semantically meaningful relationship types while preserving the original relation string as a property.

In [10]:
relationship_preview = pd.read_csv(relationships_output, usecols=["relation", "display_relation"])
relation_type_map = (
    relationship_preview.drop_duplicates()
    .sort_values(["relation", "display_relation"])
    .reset_index(drop=True)
)
relation_type_map["neo4j_relationship_type"] = relation_type_map["relation"].map(to_cypher_rel_type)

relation_type_output = NEO4J_IMPORT_DIR / "relation_type_map.csv"
relation_type_map.to_csv(relation_type_output, index=False)

relation_type_map

,relation,display_relation,neo4j_relationship_type
0,anatomy_anatomy,parent-child,ANATOMY_ANATOMY
1,anatomy_protein_absent,expression absent,ANATOMY_PROTEIN_ABSENT
2,anatomy_protein_present,expression present,ANATOMY_PROTEIN_PRESENT
3,bioprocess_bioprocess,parent-child,BIOPROCESS_BIOPROCESS
4,bioprocess_protein,interacts with,BIOPROCESS_PROTEIN
5,cellcomp_cellcomp,parent-child,CELLCOMP_CELLCOMP
6,cellcomp_protein,interacts with,CELLCOMP_PROTEIN
7,contraindication,contraindication,CONTRAINDICATION
8,disease_disease,parent-child,DISEASE_DISEASE
9,disease_phenotype_negative,phenotype absent,DISEASE_PHENOTYPE_NEGATIVE


## 11. Verify Staging Outputs

Confirm that all staging outputs exist and summarize their sizes. These files are the inputs to the Neo4j loading notebook.

In [11]:
staging_files = [nodes_output, relationships_output, relation_type_output]
summary = pd.DataFrame(
    {
        "file": [path.name for path in staging_files],
        "path": [str(path) for path in staging_files],
        "size_mb": [round(path.stat().st_size / (1024**2), 2) for path in staging_files],
    }
)
summary

,file,path,size_mb
0,nodes_enriched.csv,build/neo4j_import/nodes_enriched.csv,73.37
1,relationships_undirected.csv,build/neo4j_import/relationships_undirected.csv,192.06
2,relation_type_map.csv,build/neo4j_import/relation_type_map.csv,0.00
